# Optimization Space IIIb: MLP Trainability

<br>

## Learning Objectives and Lesson Structure

This notebook continues `03_optimization_space.ipynb`, but now the selected function is a small MLP trained on the same tilt-power sample function used across the workshop.

The learning frame is still:

$$
(\mathcal{D},\mathcal{H}_{\mathrm{MLP}},\mathcal{O})\mapsto s.
$$

Here $\mathcal{D}$ is the observed tilt-power sample, $\mathcal{H}_{\mathrm{MLP}}$ is a one-hidden-layer ReLU MLP, $\mathcal{O}$ is the training recipe, and $s=h_{\theta_T}$ is the selected realised function.

The central question is:

$$
\text{When does MLP training get useful gradients, and when does it get stuck?}
$$

By the end, students should be able to:

- read initialisation as a starting function, not only as parameter values;
- diagnose dead or symmetric starts using activation and gradient diagnostics;
- explain why raw input scale changes the effective optimisation problem;
- show why standardised inputs plus suitable initialisation improve trainability;
- compare selected MLP functions across seeds.

The notebook is organised around five questions:

1. **What shared sample function are we fitting?**
2. **What function does initialisation select?**
3. **How can initialisation get stuck?**
4. **Why do scaling and initialisation belong together?**
5. **How variable are selected MLP functions across seeds?**

The recurring researcher-facing question is:

$$
\text{Which training choices made this MLP function learnable?}
$$

<br>

### Setup

Run this cell first. It imports the shared tilt-power data helper and the compact MLP optimisation helpers used below. The examples keep the data and architecture mostly fixed while changing input scaling and initialisation.

In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

import matplotlib.pyplot as plt
import numpy as np

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "workshop1",
                "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git",
                str(repo_dir),
            ],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / "nextgen2026_mlai_workshops").exists():
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name.startswith("nextgen2026_mlai_workshops"):
        del sys.modules[module_name]

from nextgen2026_mlai_workshops.optimization_space import (
    COLORS,
    configure_optimization_matplotlib,
    display_table,
    forward_mlp,
    make_mlp_tilt_power_data,
    plot_mlp_tilt_power_fit,
    train_mlp_tilt_power_recipe,
)

configure_optimization_matplotlib()

print("Setup complete. MLP optimisation helpers imported.")

<br>

## 1. What Shared Sample Function Are We Fitting?

### Motivation

The other notebooks use a one-dimensional tilt-power setting. The latent teaching curve is:

$$
y=f_0(\theta),
$$

where $\theta$ is the true tilt angle and $Y$ is observed power. The learner sees observed samples $(X_i,Y_i)$, not the whole curve.

For this notebook, the same raw observed input can be passed to the MLP in two coordinate systems:

$$
x_{\mathrm{raw}}=X,
\qquad
x_{\mathrm{scaled}}=\frac{X-\bar X}{s_X}.
$$

The represented function is plotted back on the raw tilt axis either way. Scaling only changes the coordinates seen by optimisation.

### Minimal example

The first plot shows the shared tilt-power sample and latent reference curve. The second plot shows the same observed inputs after standardisation.

Look for the difference between:

- the scientific input scale, which is still observed tilt from $0$ to $90$;
- the numerical input scale, which is what gradient descent uses internally.

In [ ]:
# Change these values, then rerun this cell.
n = 80
data_seed = 42
sampling = "sparse_feature"      # "uniform", "sparse_feature", "dense_feature", "gap65"
y_noise = 0.045

sample = make_mlp_tilt_power_data(
    n=n,
    seed=data_seed,
    sampling=sampling,
    y_noise=y_noise,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].scatter(
    sample["x_raw"][:, 0],
    sample["y"][:, 0],
    s=28,
    color=COLORS["data"],
    edgecolor="white",
    linewidth=0.4,
    alpha=0.85,
    label="observed sample",
)
axes[0].plot(sample["grid_raw"][:, 0], sample["y_true"][:, 0], color=COLORS["truth"], lw=2.2, label="latent f0")
axes[0].axvspan(60, 70, color=COLORS["support"], alpha=0.28, lw=0, label="narrow feature")
axes[0].set_xlim(0, 90)
axes[0].set_ylim(-0.2, 1.45)
axes[0].set_title("Shared tilt-power sample")
axes[0].set_xlabel("observed tilt X")
axes[0].set_ylabel("observed power Y")
axes[0].legend(fontsize=8)

axes[1].scatter(sample["x_raw"][:, 0], sample["x_scaled"][:, 0], s=24, color=COLORS["selected"], alpha=0.82)
axes[1].axhline(0.0, color=COLORS["truth"], lw=1.2, ls="--")
axes[1].set_xlim(0, 90)
axes[1].set_title("Coordinate used by the optimiser")
axes[1].set_xlabel("observed tilt X")
axes[1].set_ylabel("standardised MLP input")

plt.tight_layout()
plt.show()

display_table(
    ["Quantity", "Value"],
    [
        ("n", n),
        ("sampling", sampling),
        ("raw X mean", sample["x_mean"]),
        ("raw X std", sample["x_std"]),
        ("scaled X mean", float(np.mean(sample["x_scaled"]))),
        ("scaled X std", float(np.std(sample["x_scaled"]))),
    ],
)

<br>

## 2. What Function Does Initialisation Select?

### Motivation

Before training begins, the MLP already represents a function:

$$
h_{\theta_0}(x)=W_2\sigma(W_1x+b_1)+b_2.
$$

Initialisation therefore selects both a starting function and the derivative information available to the optimiser.

The optimiser sees one large parameter vector, but the code stores it as arrays such as $W_1$, $b_1$, $W_2$, and $b_2$. The reported full gradient norm is:

$$
\|\nabla_\theta J(\theta_0)\|_2
=\sqrt{\sum_k \|\nabla_{\theta_k}J(\theta_0)\|_F^2},
$$

where $\theta_k$ ranges over the parameter arrays. The hidden-layer gradient norm restricts this sum to the first-layer arrays, which helps distinguish "the output bias can move" from "the hidden features can learn."

For a ReLU hidden layer, an input is active when:

$$
W_1x+b_1>0.
$$

If many units are inactive on the data, gradients through the hidden layer can disappear.

### Minimal example

The plot shows the initial function, hidden preactivations, and hidden activations before any training update.

Look for the difference between:

- initial loss;
- the visible starting function;
- active ReLU fraction;
- hidden-layer gradient norm.

In [ ]:
# Change these values, then rerun this cell.
width = 24
activation = "relu"          # "relu", "tanh"
input_mode = "scaled"        # "scaled", "raw"
init_mode = "he"             # "small", "xavier", "he", "large", "zero"
init_scale = 1.0
bias_shift = 0.0
seed = 0

init_run = train_mlp_tilt_power_recipe(
    sample,
    input_mode=input_mode,
    width=width,
    activation=activation,
    init_mode=init_mode,
    init_scale=init_scale,
    bias_shift=bias_shift,
    seed=seed,
    learning_rate=0.03,
    epochs=0,
)

x_grid = sample["grid_scaled"] if input_mode == "scaled" else sample["grid_raw"]
_, cache_grid = forward_mlp(init_run["params"], x_grid, activation=activation)

diag = init_run["initial_diagnostics"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
plot_mlp_tilt_power_fit(
    axes[0],
    sample,
    init_run["params"],
    input_mode=input_mode,
    activation=activation,
    label="initial MLP",
    color=COLORS["selected"],
)
axes[0].set_title("Initial realised function")

axes[1].hist(cache_grid["z1"].ravel(), bins=30, color=COLORS["data"], alpha=0.85)
axes[1].axvline(0.0, color=COLORS["truth"], linewidth=1.4)
axes[1].set_title("Hidden preactivations")
axes[1].set_xlabel("z1 = W1x + b1")
axes[1].set_ylabel("count")

axes[2].hist(cache_grid["a1"].ravel(), bins=30, color=COLORS["alt"], alpha=0.85)
axes[2].set_title("Hidden activations")
axes[2].set_xlabel("a1 = activation(z1)")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.show()

display_table(
    ["Diagnostic", "Value"],
    [
        ("initial train MSE", diag["train_mse"]),
        ("initial oracle grid MSE", diag["grid_oracle_mse"]),
        ("output std on grid", diag["output_std"]),
        ("full gradient norm", diag["grad_norm"]),
        ("hidden-layer gradient norm", diag["hidden_grad_norm"]),
        ("active/unsaturated fraction", diag["activity"]),
    ],
)

<br>

## 3. How Can Initialisation Get Stuck?

### Motivation

Some starting points give the optimiser little useful hidden-layer signal. Two common teaching failures are:

$$
\theta_0=0
$$

which makes hidden units symmetric, and a ReLU layer whose preactivations are negative for all training inputs:

$$
W_1x_i+b_1\leq 0\quad\text{for all }i.
$$

In both cases, the output bias may still move, but the hidden representation can remain unused. The fitted curve becomes nearly constant.

### Minimal example

The cell compares two stuck recipes on the same scaled input:

- `zero init`: all hidden units start identically;
- `dead ReLU bias`: a negative bias turns off every ReLU unit on the sample.

Look for low output variation, zero hidden-layer gradient, and a flat selected function.

In [ ]:
# Change these values, then rerun this cell.
stuck_epochs = 800
stuck_learning_rate = 0.03

stuck_recipes = [
    {
        "label": "zero init",
        "input_mode": "scaled",
        "init_mode": "zero",
        "init_scale": 1.0,
        "bias_shift": 0.0,
        "seed": 0,
        "color": COLORS["alt"],
    },
    {
        "label": "dead ReLU bias",
        "input_mode": "scaled",
        "init_mode": "small",
        "init_scale": 1.0,
        "bias_shift": -5.0,
        "seed": 0,
        "color": COLORS["regularised"],
    },
]

stuck_runs = []
for recipe in stuck_recipes:
    run = train_mlp_tilt_power_recipe(
        sample,
        input_mode=recipe["input_mode"],
        width=width,
        activation="relu",
        init_mode=recipe["init_mode"],
        init_scale=recipe["init_scale"],
        bias_shift=recipe["bias_shift"],
        seed=recipe["seed"],
        learning_rate=stuck_learning_rate,
        epochs=stuck_epochs,
        log_every=100,
    )
    run["label"] = recipe["label"]
    run["color"] = recipe["color"]
    stuck_runs.append(run)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for i, run in enumerate(stuck_runs):
    plot_mlp_tilt_power_fit(
        axes[0],
        sample,
        run["params"],
        input_mode=run["input_mode"],
        activation=run["activation"],
        label=run["label"],
        color=run["color"],
        show_data=(i == 0),
        show_truth=(i == 0),
        show_feature=(i == 0),
    )
axes[0].set_title("Selected functions after stuck training")

for run in stuck_runs:
    axes[1].plot(run["history"]["epoch"], run["history"]["loss"], color=run["color"], lw=2.1, label=run["label"])
axes[1].set_yscale("log")
axes[1].set_title("Training loss")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("train MSE")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

rows = []
for run in stuck_runs:
    final = run["final_diagnostics"]
    rows.append([
        run["label"],
        final["train_mse"],
        final["grid_oracle_mse"],
        final["output_std"],
        final["hidden_grad_norm"],
        final["activity"],
    ])

display_table(
    ["Recipe", "Final train MSE", "Oracle grid MSE", "Output std", "Hidden grad norm", "Active fraction"],
    rows,
)

<br>

## 4. Why Do Scaling and Initialisation Belong Together?

### Motivation

The same initialisation formula means different things on different input scales. With raw tilt values near $0$ to $90$, the first-layer preactivations and gradients can be much larger than they are on standardised inputs.

Gradient descent updates by:

$$
\theta_{t+1}=\theta_t-\eta\nabla_\theta J(\theta_t).
$$

For the MLP helpers, this is implemented as the same full-batch update applied to several parameter arrays:

```text
params = initialise_mlp(seed, init_mode, init_scale)
for epoch in range(T):
    yhat, cache = forward_mlp(params, x)
    grads = backward_mlp(params, cache, y)
    for each parameter array:
        params[name] = params[name] - eta * grads[name]
    log full-data loss and gradient norm
```

`train_mlp_tilt_power_recipe` chooses either raw or standardised inputs before this loop. Changing `input_mode` therefore changes the cached preactivations and the back-propagated gradients, even though the final fitted function is plotted back on the same raw tilt axis.

The learning rate $\eta$ is meaningful only relative to the gradient scale. A recipe that works on standardised inputs can collapse or behave like a constant model on raw inputs.

### Minimal example

The cell trains the same MLP recipe twice:

- once on raw observed tilt values;
- once on standardised tilt values.

Both fitted functions are plotted back on the raw scientific axis.

Look for how input scaling changes initial gradient norm, final output variation, and oracle-grid error.

In [ ]:
# Change these values, then rerun this cell.
scaling_epochs = 800
scaling_learning_rate = 0.03
scaling_seed = 0

scaling_recipes = [
    {"label": "raw input + He init", "input_mode": "raw", "color": COLORS["alt"]},
    {"label": "scaled input + He init", "input_mode": "scaled", "color": COLORS["selected"]},
]

scaling_runs = []
for recipe in scaling_recipes:
    run = train_mlp_tilt_power_recipe(
        sample,
        input_mode=recipe["input_mode"],
        width=width,
        activation="relu",
        init_mode="he",
        init_scale=1.0,
        bias_shift=0.0,
        seed=scaling_seed,
        learning_rate=scaling_learning_rate,
        epochs=scaling_epochs,
        log_every=100,
    )
    run["label"] = recipe["label"]
    run["color"] = recipe["color"]
    scaling_runs.append(run)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for i, run in enumerate(scaling_runs):
    plot_mlp_tilt_power_fit(
        axes[0],
        sample,
        run["params"],
        input_mode=run["input_mode"],
        activation=run["activation"],
        label=run["label"],
        color=run["color"],
        show_data=(i == 0),
        show_truth=(i == 0),
        show_feature=(i == 0),
    )
axes[0].set_title("Same recipe, different input coordinates")

for run in scaling_runs:
    axes[1].plot(run["history"]["epoch"], run["history"]["loss"], color=run["color"], lw=2.1, label=run["label"])
axes[1].set_yscale("log")
axes[1].set_title("Training loss")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("train MSE")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

rows = []
for run in scaling_runs:
    initial = run["initial_diagnostics"]
    final = run["final_diagnostics"]
    rows.append([
        run["label"],
        initial["train_mse"],
        initial["grad_norm"],
        final["train_mse"],
        final["grid_oracle_mse"],
        final["output_std"],
        final["hidden_grad_norm"],
    ])

display_table(
    ["Recipe", "Initial train MSE", "Initial grad norm", "Final train MSE", "Oracle grid MSE", "Output std", "Hidden grad norm"],
    rows,
)

<br>

## 5. How Variable Are Selected MLP Functions Across Seeds?

### Motivation

After scaling and initialisation are sensible, random seed still matters. A seed samples a different starting point:

$$
\theta_0^{(r)}\sim P_{\mathrm{init}}.
$$

Training then follows a different path through parameter space and may select a different realised function:

$$
s^{(r)}=h_{\theta_T^{(r)}}.
$$

The important object to compare is the selected function, not only the final loss.

### Minimal example

The cell trains several scaled-input MLPs with the same recipe and different seeds.

Look for:

- whether the functions agree where the data are dense;
- whether they differ near the narrow feature;
- how much final error varies across seeds.

In [ ]:
# Change these values, then rerun this cell.
seed_values = [0, 1, 2, 3, 4, 5]
seed_epochs = 800
seed_learning_rate = 0.03

seed_runs = []
for current_seed in seed_values:
    run = train_mlp_tilt_power_recipe(
        sample,
        input_mode="scaled",
        width=width,
        activation="relu",
        init_mode="he",
        init_scale=1.0,
        bias_shift=0.0,
        seed=current_seed,
        learning_rate=seed_learning_rate,
        epochs=seed_epochs,
        log_every=100,
    )
    run["label"] = f"seed {current_seed}"
    seed_runs.append(run)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for i, run in enumerate(seed_runs):
    color = plt.cm.tab10(i % 10)
    plot_mlp_tilt_power_fit(
        axes[0],
        sample,
        run["params"],
        input_mode=run["input_mode"],
        activation=run["activation"],
        label=run["label"],
        color=color,
        show_data=(i == 0),
        show_truth=(i == 0),
        show_feature=(i == 0),
    )
axes[0].set_title("Selected functions across seeds")

for i, run in enumerate(seed_runs):
    axes[1].plot(run["history"]["epoch"], run["history"]["loss"], color=plt.cm.tab10(i % 10), lw=1.8, label=run["label"])
axes[1].set_yscale("log")
axes[1].set_title("Training loss across seeds")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("train MSE")
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

seed_rows = []
for run in seed_runs:
    final = run["final_diagnostics"]
    seed_rows.append([
        run["seed"],
        final["train_mse"],
        final["grid_oracle_mse"],
        final["output_std"],
        final["activity"],
    ])

grid_errors = np.array([run["final_diagnostics"]["grid_oracle_mse"] for run in seed_runs])

display_table(
    ["Seed", "Final train MSE", "Oracle grid MSE", "Output std", "Active fraction"],
    seed_rows,
)

display_table(
    ["Audit item", "Choice in this notebook", "Why it matters"],
    [
        ("Data", "shared tilt-power sample", "defines the observed evidence"),
        ("Architecture", f"one hidden layer, width={width}, ReLU", "defines available MLP functions"),
        ("Input scaling", "raw versus standardised", "changes preactivation and gradient scale"),
        ("Initialisation", "zero, dead-bias, He", "chooses starting function and hidden gradient geometry"),
        ("Learning rate", seed_learning_rate, "only meaningful relative to gradient scale"),
        ("Stopping", f"{seed_epochs} epochs", "selects the reported iterate"),
        ("Seed variability", float(np.std(grid_errors)), "shows remaining function-selection sensitivity"),
    ],
)

### Final Takeaway

For an MLP, optimisation is not only the command "minimise the loss." It is a training recipe that chooses a starting function, a numerical input scale, an update size, and a final iterate.

The compact audit is:

| Question | Main diagnostic | Research concern |
|---|---|---|
| What sample function is being fitted? | shared tilt-power data and $f_0$ | evidence and claim context |
| What did initialisation select? | initial function, activity, gradient norm | starting function |
| Did training get stuck? | output std, hidden gradient norm, active fraction | trainability |
| Did scaling help? | raw vs standardised input comparison | stable optimisation |
| Do seeds agree? | selected functions and grid error across seeds | selection stability |

The practical question is:

$$
\text{Could this MLP have learned the claimed behaviour under the reported training recipe?}
$$